# Qdrant 벡터 청크 분석 노트북

이 노트북은 Qdrant에 저장된 문서 벡터 청크의 메타데이터를 수집하고, 시각화를 통해 분석하는 데 사용됩니다.

In [ ]:
from qdrant_client import QdrantClient
import pandas as pd
import matplotlib.pyplot as plt

client = QdrantClient(host="169.211.207.99", port=1031)
collection_name = "documents"

# 컬렉션 목록 및 문서 수 확인
print(client.get_collections())
print(client.count(collection_name=collection_name, exact=True))


In [ ]:
result = client.scroll(
    collection_name=collection_name,
    limit=1000,
    with_payload=True
)
points = result[0]

data = []
for p in points:
    meta = p.payload or {}
    inner = meta.get("metadata", {})
    content = meta.get("page_content", "N/A")

    data.append({
        "document_id": inner.get("document_id", "unknown"),
        "email": inner.get("email", "unknown"),
        "filename": inner.get("filename", "unknown"),
        "length": len(content),
        "preview": content[:100].replace("\n", " ") + "..."
    })

df = pd.DataFrame(data)
df.head()


In [ ]:
df['document_id'] = df['document_id'].astype(str)
df.groupby("document_id").size().plot(kind='bar', title="청크 개수 by Document ID", figsize=(10, 4))
plt.ylabel("Chunk Count")
plt.xlabel("Document ID")
plt.tight_layout()
plt.show()


In [ ]:
df["length"].plot.hist(bins=30, title="청크 길이 분포", figsize=(8, 4))
plt.xlabel("Text Length (chars)")
plt.tight_layout()
plt.show()


In [ ]:
df[["document_id", "filename", "length", "preview"]].head(10)
